In [ ]:
"""
Var, Cov, Cor, Cosine similarity - 一次串清楚
"""
import numpy as np

np.set_printoptions(precision=4, suppress=True)

# 兩個變數: 念書時數 vs 考試分數
x = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=float)
y = np.array([55, 60, 70, 65, 80, 75, 90, 95], dtype=float)
n = len(x)

# ============================================================
# 1. VARIANCE — 單一變數的散佈
# ============================================================
var_x_manual = np.sum((x - x.mean())**2) / (n - 1)
print(f"Var(X) 手算:  {var_x_manual:.4f}")
print(f"Var(X) numpy: {np.var(x, ddof=1):.4f}")
# 注意 ddof=1: 樣本 variance 除 (n-1)，不是 n

# ============================================================
# 2. COVARIANCE — 兩變數共同變化
# ============================================================
cov_xy_manual = np.sum((x - x.mean()) * (y - y.mean())) / (n - 1)
print(f"\nCov(X,Y) 手算:  {cov_xy_manual:.4f}")
print(f"Cov(X,Y) numpy: {np.cov(x, y, ddof=1)[0, 1]:.4f}")
# np.cov 回傳 2x2 矩陣: [[Var(X), Cov(X,Y)], [Cov(X,Y), Var(Y)]]

# ============================================================
# 3. CORRELATION — covariance 除以兩個 std
# ============================================================
cor_manual = cov_xy_manual / (np.sqrt(np.var(x, ddof=1)) * np.sqrt(np.var(y, ddof=1)))
print(f"\nCor(X,Y) 手算:  {cor_manual:.4f}")
print(f"Cor(X,Y) numpy: {np.corrcoef(x, y)[0, 1]:.4f}")

# ============================================================
# 4. COSINE SIMILARITY — 兩向量的夾角 這就會是slide 裡面的 similarity score，分母部分相成再開根，就是絕對值
# ============================================================
# (a) 直接對原始資料: 沒 center
cos_raw = np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y))
print(f"\nCosine similarity (原始):     {cos_raw:.4f}")

# (b) 對 centered 資料: 這個會等於 correlation
xc = x - x.mean()
yc = y - y.mean()
cos_centered = np.dot(xc, yc) / (np.linalg.norm(xc) * np.linalg.norm(yc))
print(f"Cosine similarity (centered): {cos_centered:.4f}")
print(f"→ 跟 Cor(X,Y) 一樣! correlation 就是 centered 後的 cosine similarity")

# ============================================================
# 5. 為什麼用 correlation 不用 covariance — 尺度不變
# ============================================================
y_scaled = y * 1000 + 5000   # 任意線性變換 (改單位)
print(f"\n--- 把 Y 乘 1000 再加 5000 ---")
print(f"Cov(X, Y_scaled):  {np.cov(x, y_scaled, ddof=1)[0,1]:.2f}   ← 暴增!")
print(f"Cor(X, Y_scaled):  {np.corrcoef(x, y_scaled)[0,1]:.4f}   ← 完全沒變!")
print("→ correlation 是 scale-invariant，所以才好比較不同單位的變數")

Var(X) 手算:  6.0000
Var(X) numpy: 6.0000

Cov(X,Y) 手算:  32.8571
Cov(X,Y) numpy: 32.8571

Cor(X,Y) 手算:  0.9528
Cor(X,Y) numpy: 0.9528

Cosine similarity (原始):     0.9533
Cosine similarity (centered): 0.9528
→ 跟 Cor(X,Y) 一樣! correlation 就是 centered 後的 cosine similarity

--- 把 Y 乘 1000 再加 5000 ---
Cov(X, Y_scaled):  32857.14   ← 暴增!
Cor(X, Y_scaled):  0.9528   ← 完全沒變!
→ correlation 是 scale-invariant，所以才好比較不同單位的變數


In [2]:
"""
Total vs Generalized sample variance — 同樣的 total，差很大的 generalized
"""
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)
n = 300

# ============================================================
# Dataset A: 球形 (variables 無相關)
# ============================================================
cov_A = np.array([[1.0, 0.0],
                  [0.0, 1.0]])
X_A = rng.multivariate_normal([0, 0], cov_A, size=n)

# ============================================================
# Dataset B: 高度相關 (扁長橢圓)
# ============================================================
cov_B = np.array([[1.0, 0.95],
                  [0.95, 1.0]])
X_B = rng.multivariate_normal([0, 0], cov_B, size=n)

# ============================================================
# 計算兩個 summary 的數字
# ============================================================
S_A = np.cov(X_A, rowvar=False)
S_B = np.cov(X_B, rowvar=False)

print("Dataset A (球形):")
print(f"  S_A =\n{S_A}")
print(f"  Total variance     tr(S_A) = {np.trace(S_A):.4f}")
print(f"  Generalized var.   |S_A|   = {np.linalg.det(S_A):.4f}")
print(f"  Eigenvalues:       {np.linalg.eigvalsh(S_A)}")

print("\nDataset B (扁長):")
print(f"  S_B =\n{S_B}")
print(f"  Total variance     tr(S_B) = {np.trace(S_B):.4f}")
print(f"  Generalized var.   |S_B|   = {np.linalg.det(S_B):.4f}")
print(f"  Eigenvalues:       {np.linalg.eigvalsh(S_B)}")

print("\n→ Total variance 幾乎一樣 (~2)")
print("→ Generalized variance 差超多: A 接近 1, B 接近 0.1")
print("→ 因為 B 有一個 eigenvalue 很小 (扁掉的方向)")

# ============================================================
# 視覺化
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, X, S, title in [(axes[0], X_A, S_A, "Dataset A: spherical"),
                         (axes[1], X_B, S_B, "Dataset B: correlated (flat)")]:
    ax.scatter(X[:, 0], X[:, 1], alpha=0.5, s=15)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', lw=0.5)
    ax.axvline(0, color='k', lw=0.5)
    ax.set_title(f"{title}\n"
                 f"tr(S) = {np.trace(S):.3f}, |S| = {np.linalg.det(S):.4f}")

plt.tight_layout()
plt.savefig('/Users/fangsiyu/Desktop/sdu-2026-code/805_multivariate_statistical_analysis/note/fig4_total_vs_generalized.png', dpi=110, bbox_inches='tight')
plt.close()
print("\n→ 圖存到 fig_total_vs_generalized.png")

Dataset A (球形):
  S_A =
[[0.9951 0.1396]
 [0.1396 0.9047]]
  Total variance     tr(S_A) = 1.8997
  Generalized var.   |S_A|   = 0.8807
  Eigenvalues:       [0.8032 1.0966]

Dataset B (扁長):
  S_B =
[[1.01   0.9071]
 [0.9071 0.9081]]
  Total variance     tr(S_B) = 1.9180
  Generalized var.   |S_B|   = 0.0943
  Eigenvalues:       [0.0505 1.8675]

→ Total variance 幾乎一樣 (~2)
→ Generalized variance 差超多: A 接近 1, B 接近 0.1
→ 因為 B 有一個 eigenvalue 很小 (扁掉的方向)

→ 圖存到 fig_total_vs_generalized.png


In [5]:
"""
VIF vs det(R) vs condition number — 三個工具看同一個共線問題
"""
import numpy as np

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)
n = 200

# 故意做出共線: x3 ≈ 0.95 * x1
x1 = rng.normal(0, 1, n)
x2 = rng.normal(0, 1, n)
x3 = 0.95 * x1 + 0.1 * rng.normal(0, 1, n)   # 高度相關 with x1
x4 = rng.normal(0, 1, n)

X = np.column_stack([x1, x2, x3, x4])
names = ['x1', 'x2', 'x3', 'x4']

# ============================================================
# Method 1: VIF — 一個變數一個變數看
# ============================================================
def compute_vif(X, k):
    """把第 k 行當 y, 其他行當 X 跑迴歸, 算 R²"""
    others = np.column_stack([np.ones(len(X)), np.delete(X, k, axis=1)])
    beta, *_ = np.linalg.lstsq(others, X[:, k], rcond=None)
    y_hat = others @ beta
    r_sq = 1 - np.sum((X[:, k] - y_hat)**2) / np.sum((X[:, k] - X[:, k].mean())**2)
    return 1 / (1 - r_sq), r_sq

print("=" * 50)
print("Method 1: VIF (per-variable diagnostic)")
print("=" * 50)
for k in range(4):
    vif, r_sq = compute_vif(X, k)
    flag = "  ⚠ HIGH" if vif > 10 else ""
    print(f"  {names[k]}: R² = {r_sq:.4f}, VIF = {vif:7.3f}{flag}")
print("→ x1 跟 x3 互相預測到 R² ~ 0.9, VIF 爆炸")

# ============================================================
# Method 2: det of correlation matrix — 全域
# ============================================================
R = np.corrcoef(X, rowvar=False)
print(f"\n" + "=" * 50)
print("Method 2: det(R) (global diagnostic)")
print("=" * 50)
print(f"Correlation matrix R:\n{R}")
print(f"\ndet(R) = {np.linalg.det(R):.6f}")
print(f"→ 接近 0 → 矩陣『接近 singular』→ 有共線")
print(f"→ 但 det 只說『有問題』, 不告訴你『是哪個變數』")



Method 1: VIF (per-variable diagnostic)
  x1: R² = 0.9855, VIF =  69.014  ⚠ HIGH
  x2: R² = 0.0078, VIF =   1.008
  x3: R² = 0.9855, VIF =  68.941  ⚠ HIGH
  x4: R² = 0.0104, VIF =   1.011
→ x1 跟 x3 互相預測到 R² ~ 0.9, VIF 爆炸

Method 2: det(R) (global diagnostic)
Correlation matrix R:
[[ 1.     -0.0702  0.9927 -0.0238]
 [-0.0702  1.     -0.066  -0.0396]
 [ 0.9927 -0.066   1.     -0.0129]
 [-0.0238 -0.0396 -0.0129  1.    ]]

det(R) = 0.014401
→ 接近 0 → 矩陣『接近 singular』→ 有共線
→ 但 det 只說『有問題』, 不告訴你『是哪個變數』


In [6]:
# ============================================================
# Method 3: Condition number — eigenvalue-based
# ============================================================
eigvals = np.linalg.eigvalsh(R)
print(f"\n" + "=" * 50)
print("Method 3: Condition number (eigenvalue ratio)")
print("=" * 50)
print(f"Eigenvalues of R: {eigvals}")
print(f"  最小 eigenvalue: {eigvals[0]:.4f}  ← 接近 0 → 有一個方向資訊量超少")
print(f"  最大 eigenvalue: {eigvals[-1]:.4f}")
print(f"  Condition number κ = {eigvals[-1]/eigvals[0]:.2f}")
print(f"→ κ > 30 通常被認為有共線; κ > 100 嚴重")

# ============================================================
# 對照: 丟掉 x3 後重新檢查
# ============================================================
print("\n" + "=" * 50)
print("把 x3 丟掉後重新檢查 (這就是你 report 在做的事)")
print("=" * 50)
X_clean = np.column_stack([x1, x2, x4])
names_clean = ['x1', 'x2', 'x4']
for k in range(3):
    vif, _ = compute_vif(X_clean, k)
    print(f"  {names_clean[k]}: VIF = {vif:.3f}")
R_clean = np.corrcoef(X_clean, rowvar=False)
ev_clean = np.linalg.eigvalsh(R_clean)
print(f"\ndet(R) = {np.linalg.det(R_clean):.4f}  (從 ~0 變回正常)")
print(f"κ = {ev_clean[-1]/ev_clean[0]:.2f}  (從爆炸值變回正常)")
print("→ 三個診斷工具同步『康復』, 證實它們在看同一個現象")


Method 3: Condition number (eigenvalue ratio)
Eigenvalues of R: [0.0073 0.9531 1.0373 2.0024]
  最小 eigenvalue: 0.0073  ← 接近 0 → 有一個方向資訊量超少
  最大 eigenvalue: 2.0024
  Condition number κ = 275.25
→ κ > 30 通常被認為有共線; κ > 100 嚴重

把 x3 丟掉後重新檢查 (這就是你 report 在做的事)
  x1: VIF = 1.006
  x2: VIF = 1.007
  x4: VIF = 1.002

det(R) = 0.9928  (從 ~0 變回正常)
κ = 1.18  (從爆炸值變回正常)
→ 三個診斷工具同步『康復』, 證實它們在看同一個現象
